In [ ]:
# ========== 1. IMPORTS ========== #
import os, csv, time, random, shutil, glob, gc
from collections import defaultdict
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, BatchNormalization,
    Conv2D, Activation, Input, Layer,
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, Callback
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score, matthews_corrcoef,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE

# GPU + Mixed Precision
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print(f"GPUs: {len(gpus)} | Policy: mixed_float16")

In [ ]:

# ========== 2. CONFIGURATION ========== #

STRATEGY_KEY   = "efsda_v2_freqband_temperature"
STRATEGY_LABEL = "EfficientNetB4 + E-FSDA v2 (FreqBand + Temperature)"

DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report_EfficientNetB4/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

INPUT_SHAPE     = (380, 380, 3)
BATCH_SIZE      = 32
EPOCHS          = 30
LR              = 1e-4
UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]
DROPOUT_RATE    = 0.6     # increased from 0.5 to reduce overfitting
PATIENCE        = 8       # reduced from 12 to stop earlier

# E-FSDA v2 hyperparams
EFSDA_REDUCTION  = 16
EFSDA_SPATIAL_KS = 7
EFSDA_N_BANDS    = 2     # Low/High only — reduced from 3 to limit param count on small dataset

# Loss
FOCAL_GAMMA = 2.0
CB_BETA     = 0.9999
ADAPTIVE_TAU = 0.3

# Multi-run
N_RUNS       = 3
RANDOM_SEEDS = [42, 123, 456]
AUTOTUNE     = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

print(f"Strategy: {STRATEGY_LABEL}")
print(f"Novel: Frequency Band Decomposition ({EFSDA_N_BANDS} bands) + Learnable Temperature")
print(f"Unfreeze: {UNFREEZE_BLOCKS} | LR: {LR} | BS: {BATCH_SIZE} | Dropout: {DROPOUT_RATE}")


In [ ]:

# ========== 3. E-FSDA v2: ENHANCED FREQUENCY-SPATIAL DUAL ATTENTION ========== #

class FrequencyBandAttention(Layer):
    """Enhanced Frequency Channel Attention with Band Decomposition.
    
    Novel improvements over standard FrequencyChannelAttention:
    1. FREQUENCY BAND DECOMPOSITION — Low/Mid/High bands with learnable importance
    2. LEARNABLE TEMPERATURE SCALING — controls attention sharpness

    All call() computation in float32; weights AND intermediates cast at use-site
    to survive mixed_float16 policy + JIT (set_jit(True) may re-cast even after
    tf.cast, so we cast each operand individually right before each op).
    """
    def __init__(self, reduction=16, n_bands=3, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.n_bands = n_bands

    def build(self, input_shape):
        C = input_shape[-1]
        H, W = input_shape[1], input_shape[2]
        r = max(C // self.reduction, 8)

        self.band_fc1 = []
        self.band_fc2 = []
        for i in range(self.n_bands):
            fc1 = Dense(r, use_bias=False, dtype='float32', name=f'{self.name}_band{i}_fc1')
            fc2 = Dense(C, use_bias=False, dtype='float32', name=f'{self.name}_band{i}_fc2')
            fc1.build((None, C))
            fc2.build((None, r))
            self.band_fc1.append(fc1)
            self.band_fc2.append(fc2)

        self.band_weights = self.add_weight(
            name='band_importance', shape=(self.n_bands,),
            initializer=tf.keras.initializers.Constant(1.0 / self.n_bands),
            trainable=True)

        self.temperature = self.add_weight(
            name='freq_temperature', shape=(1, C),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())

        if H is not None and W is not None:
            self._build_freq_masks(H, W)

        super().build(input_shape)

    def _build_freq_masks(self, H, W):
        freq_y = np.fft.fftfreq(H).reshape(-1, 1)
        freq_x = np.fft.fftfreq(W).reshape(1, -1)
        radius = np.sqrt(freq_y**2 + freq_x**2)
        max_r = radius.max()
        band_edges = np.linspace(0, max_r, self.n_bands + 1)
        masks = []
        for i in range(self.n_bands):
            mask = ((radius >= band_edges[i]) & (radius < band_edges[i+1])).astype(np.float32)
            if mask.sum() == 0:
                mask = np.ones_like(mask) / (H * W)
            masks.append(mask)
        self.freq_masks = [tf.constant(m, dtype=tf.float32) for m in masks]

    def call(self, x, training=False):
        # Force everything to float32 — policy + JIT may silently re-cast
        x_f32 = tf.cast(x, tf.float32)

        x_t = tf.transpose(x_f32, [0, 3, 1, 2])
        x_complex = tf.complex(x_t, tf.zeros_like(x_t))
        x_fft = tf.signal.fft2d(x_complex)
        mag = tf.math.log1p(tf.abs(x_fft))  # float32

        band_descriptors = []
        for i in range(self.n_bands):
            masked_mag = tf.cast(mag, tf.float32) * tf.cast(self.freq_masks[i][tf.newaxis, tf.newaxis, :, :], tf.float32)
            band_desc = tf.reduce_mean(masked_mag, axis=[2, 3])          # float32
            band_desc = tf.cast(band_desc, tf.float32)
            band_desc = tf.nn.relu(self.band_fc1[i](band_desc))
            band_desc = self.band_fc2[i](band_desc)
            band_descriptors.append(tf.cast(band_desc, tf.float32))     # explicit float32

        # Compute softmax weights — cast AFTER softmax to guard against JIT re-cast
        band_w_raw = tf.cast(self.band_weights, tf.float32)
        band_w = tf.cast(tf.nn.softmax(band_w_raw), tf.float32)         # shape (n_bands,)

        # Accumulate with explicit per-element casts
        fused = tf.zeros(tf.shape(band_descriptors[0]), dtype=tf.float32)
        for i in range(self.n_bands):
            bw_i = tf.cast(band_w[i], tf.float32)
            bd_i = tf.cast(band_descriptors[i], tf.float32)
            fused = fused + bw_i * bd_i

        temperature = tf.nn.softplus(tf.cast(self.temperature, tf.float32)) + 1e-6
        attn = tf.nn.sigmoid(fused / temperature)
        attn = tf.reshape(attn, [tf.shape(x_f32)[0], 1, 1, tf.shape(x_f32)[3]])
        out = x_f32 * tf.cast(attn, tf.float32)
        return tf.cast(out, x.dtype)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'reduction': self.reduction, 'n_bands': self.n_bands})
        return cfg


class EFSDAv2Block(Layer):
    """Enhanced Frequency-Spatial Dual Attention v2.
    Returns: (fused_features, spatial_attn_map)
    """
    def __init__(self, reduction=16, spatial_kernel=7, n_bands=3, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction
        self.spatial_kernel = spatial_kernel
        self.n_bands = n_bands

    def build(self, input_shape):
        self.freq_attn = FrequencyBandAttention(
            reduction=self.reduction, n_bands=self.n_bands,
            name=f'{self.name}_freq_band_attn')

        self.sp_conv = Conv2D(
            1, self.spatial_kernel, padding='same', use_bias=False,
            kernel_initializer='glorot_uniform', dtype='float32',
            name=f'{self.name}_sp_conv')

        self.spatial_temperature = self.add_weight(
            name='spatial_temperature', shape=(1, 1, 1, 1),
            initializer=tf.keras.initializers.Ones(),
            trainable=True, constraint=tf.keras.constraints.NonNeg())

        self.bn = BatchNormalization(dtype='float32', name=f'{self.name}_bn')

        self.freq_attn.build(input_shape)
        sp_input_shape = tuple(input_shape[:-1]) + (2,)
        self.sp_conv.build(sp_input_shape)
        self.bn.build(input_shape)
        super().build(input_shape)

    def call(self, x, training=False):
        input_dtype = x.dtype
        x_f32 = tf.cast(x, tf.float32)

        freq_out = tf.cast(self.freq_attn(x, training=training), tf.float32)

        avg_pool = tf.reduce_mean(x_f32, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(x_f32, axis=-1, keepdims=True)
        sp_logits = tf.cast(self.sp_conv(tf.concat([avg_pool, max_pool], axis=-1)), tf.float32)

        # Cast weight to float32 at use-site — both before and after softplus
        sp_temp = tf.nn.softplus(tf.cast(self.spatial_temperature, tf.float32)) + 1e-6
        sp_temp = tf.cast(sp_temp, tf.float32)
        sp_attn = tf.nn.sigmoid(sp_logits / sp_temp)

        spatial_out = x_f32 * tf.cast(sp_attn, tf.float32)
        fused = tf.cast(freq_out, tf.float32) + tf.cast(spatial_out, tf.float32)
        fused = self.bn(fused, training=training)
        fused = tf.cast(fused, input_dtype)
        return fused, sp_attn

    def compute_output_spec(self, x, training=False):
        """Bypass tf.signal.fft2d symbolic tracing for Keras 3."""
        import keras
        sp_shape = tuple(x.shape[:-1]) + (1,)
        return (
            keras.KerasTensor(x.shape, dtype=x.dtype),
            keras.KerasTensor(sp_shape, dtype='float32'),
        )

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'reduction': self.reduction,
            'spatial_kernel': self.spatial_kernel,
            'n_bands': self.n_bands,
        })
        return cfg


print("E-FSDA v2 defined.")
print("  Novel 1: Frequency Band Decomposition (Low/Mid/High)")
print("  Novel 2: Learnable Temperature Scaling (channel + spatial)")
print(f"  Params: n_bands={EFSDA_N_BANDS}, reduction={EFSDA_REDUCTION}")


In [ ]:
# ========== 4. ADAPTIVE CLASS-BALANCED FOCAL LOSS (same as baseline) ========== #

class AdaptiveClassBalancedFocalLoss(tf.keras.losses.Loss):
    def __init__(self, samples_per_class, num_classes, gamma=2.0, beta=0.9999, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.beta = beta
        self.num_classes = num_classes
        self._samples_per_class = list(samples_per_class)
        n = np.array(samples_per_class, dtype=np.float32)
        eff_num = 1.0 - np.power(beta, n)
        weights = (1.0 - beta) / eff_num
        weights = weights / weights.sum() * len(samples_per_class)
        self.static_weights = tf.constant(weights, dtype=tf.float32)
        self.adaptive_factor = tf.Variable(
            tf.ones([num_classes], dtype=tf.float32),
            trainable=False, name='adaptive_cb_factor')

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        combined_weights = self.static_weights * self.adaptive_factor
        combined_weights = combined_weights / tf.reduce_mean(combined_weights)
        sample_w = tf.reduce_sum(y_true * combined_weights, axis=-1)
        pt = tf.reduce_sum(y_true * y_pred, axis=-1)
        focal = tf.pow(1.0 - pt, self.gamma)
        ce = -tf.reduce_sum(y_true * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(sample_w * focal * ce)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'samples_per_class': self._samples_per_class,
                    'num_classes': self.num_classes,
                    'gamma': self.gamma, 'beta': self.beta})
        return cfg


class AdaptiveWeightCallback(Callback):
    def __init__(self, loss_fn, val_ds, num_classes, class_names, tau=0.3, **kwargs):
        super().__init__(**kwargs)
        self.loss_fn = loss_fn
        self.val_ds = val_ds
        self.num_classes = num_classes
        self.class_names = class_names
        self.tau = tau
        self.history = []

    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = self.model.predict(self.val_ds, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in self.val_ds])
        per_class_recall = np.zeros(self.num_classes)
        for c in range(self.num_classes):
            mask = y_true == c
            per_class_recall[c] = (y_pred[mask] == c).mean() if mask.sum() > 0 else 1.0
        epsilon = 0.1
        adaptation_target = (1.0 - per_class_recall) + epsilon
        current_factor = self.loss_fn.adaptive_factor.numpy()
        new_factor = (1.0 - self.tau) * current_factor + self.tau * adaptation_target
        new_factor = new_factor / new_factor.mean()
        self.loss_fn.adaptive_factor.assign(new_factor.astype(np.float32))
        self.history.append({'epoch': epoch+1,
                             'per_class_recall': per_class_recall.copy(),
                             'adaptive_factor': new_factor.copy()})

print("Adaptive CB Focal Loss + Callback defined.")

In [ ]:
# ========== 5. HELPER FUNCTIONS ========== #

efficientnet_preprocess = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def apply_freeze_strategy(base, unfreeze_blocks):
    base.trainable = False
    for layer in base.layers:
        for block_num in unfreeze_blocks:
            if layer.name.startswith(f"block{block_num}"):
                if not isinstance(layer, tf.keras.layers.BatchNormalization):
                    layer.trainable = True
                break
    trainable = sum(1 for l in base.layers if l.trainable)
    print(f"  Backbone: {trainable}/{len(base.layers)} layers trainable")


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, seed=None):
    class_names = sorted([d for d in os.listdir(os.path.join(data_dir, 'train'))
                          if os.path.isdir(os.path.join(data_dir, 'train', d))])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    num_classes = len(class_names)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_jpeg(raw, channels=3)
        img = tf.image.resize(img, [h, w])
        img = tf.cast(img, tf.float32)
        img = efficientnet_preprocess(img)
        return img, tf.one_hot(label, depth=num_classes)

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if training:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, len(paths), fns, labels

    train_ds, n_train, _, train_lbl = _make_split('train', training=True)
    val_ds, n_val, _, _ = _make_split('val', training=False)
    test_ds, n_test, test_fnames, test_lbl = _make_split('test', training=False)

    cw = class_weight.compute_class_weight('balanced', classes=np.unique(train_lbl), y=train_lbl)
    meta = SimpleNamespace(
        class_names=class_names, num_classes=num_classes,
        test_filenames=test_fnames, test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=dict(enumerate(cw)),
    )
    print(f"  Data: train={n_train} val={n_val} test={n_test}")
    return train_ds, val_ds, test_ds, meta


print("Helpers defined.")

In [ ]:
# ========== 6. MODEL BUILDER ========== #

CUSTOM_OBJECTS = {
    'FrequencyBandAttention': FrequencyBandAttention,
    'EFSDAv2Block': EFSDAv2Block,
    'AdaptiveClassBalancedFocalLoss': AdaptiveClassBalancedFocalLoss,
}


def build_efsda_v2_model(input_shape, num_classes, steps_per_epoch, samples_per_class):
    """EfficientNetB4 + E-FSDA v2 (FreqBand + Temperature)."""
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=input_shape)
    apply_freeze_strategy(base, UNFREEZE_BLOCKS)

    feat_map = base.output

    attended, sp_attn_map = EFSDAv2Block(
        reduction=EFSDA_REDUCTION,
        spatial_kernel=EFSDA_SPATIAL_KS,
        n_bands=EFSDA_N_BANDS,
        name='efsda_v2',
    )(feat_map)

    x = GlobalAveragePooling2D(name='gap')(attended)
    x = BatchNormalization(name='head_bn')(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-5), name='head_dense')(x)
    x = Dropout(DROPOUT_RATE, name='head_dropout')(x)
    out = Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)

    model = Model(inputs=base.input, outputs=out, name='EfficientNetB4_EFSDAv2')

    loss_fn = AdaptiveClassBalancedFocalLoss(
        samples_per_class=samples_per_class,
        num_classes=num_classes, gamma=FOCAL_GAMMA, beta=CB_BETA)

    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=LR, decay_steps=steps_per_epoch * 5,
        decay_rate=0.9, staircase=True)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
                  loss=loss_fn, metrics=['accuracy'])
    return model, loss_fn


print("Model builder defined.")

In [ ]:

# ========== 7. MULTI-RUN TRAINING ========== #

for run_idx, seed in enumerate(RANDOM_SEEDS[:N_RUNS]):
    print("\n" + "="*70)
    print(f" RUN {run_idx+1}/{N_RUNS}  seed={seed}  |  {STRATEGY_LABEL}")
    print("="*70)

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_{run_idx+1}_seed_{seed}")
    os.makedirs(RESULT_DIR, exist_ok=True)

    train_ds, val_ds, test_ds, meta = create_tf_datasets(
        DATA_DIR, INPUT_SHAPE, BATCH_SIZE, seed=seed)
    steps_per_epoch = meta.n_train // BATCH_SIZE

    samples_per_class = np.array([
        len([f for f in os.listdir(os.path.join(DATA_DIR, 'train', cn))
             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        for cn in meta.class_names
    ])

    model, loss_fn = build_efsda_v2_model(
        INPUT_SHAPE, meta.num_classes, steps_per_epoch, samples_per_class)

    if run_idx == 0:
        model.summary(print_fn=lambda x: print(x) if 'efsda' in x.lower() or 'Total' in x or 'Trainable' in x else None)

    adaptive_cb = AdaptiveWeightCallback(
        loss_fn=loss_fn, val_ds=val_ds,
        num_classes=meta.num_classes, class_names=meta.class_names, tau=ADAPTIVE_TAU)

    callbacks = [
        adaptive_cb,
        # Monitor val_accuracy for early stopping + checkpoint — avoids saving overfit low-loss models
        EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                      restore_best_weights=True, mode='max', verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'training_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'best_model.keras'),
                        save_best_only=True, monitor='val_accuracy', mode='max', verbose=1),
    ]

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks)

    # Evaluate
    best_model = load_model(os.path.join(RESULT_DIR, 'best_model.keras'),
                            custom_objects=CUSTOM_OBJECTS)
    pred_probs = best_model.predict(test_ds, verbose=0)
    y_pred_run = np.argmax(pred_probs, axis=1)
    y_true_run = meta.test_classes

    report = classification_report(y_true_run, y_pred_run,
                                   target_names=meta.class_names, output_dict=True, digits=4)
    test_acc = np.mean(y_pred_run == y_true_run)

    # Save artifacts
    with open(os.path.join(RESULT_DIR, 'classification_report.txt'), 'w') as f:
        f.write(classification_report(y_true_run, y_pred_run,
                                      target_names=meta.class_names, digits=4))

    cm = confusion_matrix(y_true_run, y_pred_run)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=meta.class_names,
                yticklabels=meta.class_names, cmap='Blues', ax=ax)
    ax.set_title(f'CM — Run {run_idx+1}'); plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrix.png'), dpi=300); plt.close()

    all_runs_results.append({
        'run': run_idx+1, 'seed': seed,
        'accuracy': test_acc,
        'precision': report['weighted avg']['precision'],
        'recall': report['weighted avg']['recall'],
        'f1_score': report['weighted avg']['f1-score'],
        'per_class_metrics': {c: report[c] for c in meta.class_names},
        'result_dir': RESULT_DIR,
        'history': history.history,
        'y_true': y_true_run, 'y_pred': y_pred_run,
        'pred_probs': pred_probs,
        'class_names': meta.class_names,
        'test_filenames': meta.test_filenames,
        'n_train': meta.n_train, 'n_val': meta.n_val, 'n_test': meta.n_test,
        'adaptive_history': adaptive_cb.history,
    })

    print(f"  Acc={test_acc:.4f}  F1={report['weighted avg']['f1-score']:.4f}")
    tf.keras.backend.clear_session()

print("\n" + "="*70 + f"\n ALL {N_RUNS} RUNS COMPLETED\n" + "="*70)


In [ ]:
# ========== 8. RESULTS AGGREGATION ========== #

accuracies = [r['accuracy'] for r in all_runs_results]
f1_scores  = [r['f1_score'] for r in all_runs_results]

print(f"\n{'='*60}")
print(f"  {STRATEGY_LABEL}")
print(f"{'='*60}")
print(f"  Accuracy : {np.mean(accuracies):.4f} ± {np.std(accuracies):.4f}")
print(f"  F1-Score : {np.mean(f1_scores):.4f} ± {np.std(f1_scores):.4f}")
print(f"  Per run  : {[f'{a:.4f}' for a in accuracies]}")

for r in all_runs_results:
    kappa = cohen_kappa_score(r['y_true'], r['y_pred'])
    mcc = matthews_corrcoef(r['y_true'], r['y_pred'])
    print(f"  Run {r['run']}: Kappa={kappa:.4f}  MCC={mcc:.4f}")

# Save summary
summary_df = pd.DataFrame([{
    'strategy': STRATEGY_KEY, 'run': r['run'], 'seed': r['seed'],
    'accuracy': r['accuracy'], 'f1_score': r['f1_score'],
} for r in all_runs_results])
summary_df.to_csv(os.path.join(BASE_RESULT_DIR, 'summary.csv'), index=False)

# Zip
zip_path = f"/kaggle/working/{STRATEGY_KEY}_complete"
shutil.make_archive(zip_path, 'zip', BASE_RESULT_DIR)
print(f"\n✅ Archived → {zip_path}.zip")